# Notebook 03 — Counterfactual replay (interactive what-ifs)

Interactive harness around R10's `CounterfactualReplayer`. Pick a
30-day window + a policy override; the replayer reads the cold
tier, re-decides every decision under the alternative policy, and
computes the hypothetical Sharpe diff vs realised paper PnL.

Target wall time per replay: < 5 min (spec § 3.4 + § 6 gate).

_Spec_: `docs/ROUND_13_CALIBRATION_AND_RESEARCH.md` § 3.5


## Parameters


In [ ]:
from datetime import datetime, timedelta, timezone
PERIOD_END = datetime.now(tz=timezone.utc).date()
PERIOD_START = PERIOD_END - timedelta(days=30)
POLICY_TO_DISABLE = 'volume_anticipation'  # or 'causal_gate'
print(f'Replay window: {PERIOD_START} → {PERIOD_END}')
print(f'Policy to disable: {POLICY_TO_DISABLE}')


## Run the replay


In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()

try:
    from src.causal.counterfactual_replay import CounterfactualReplayer
    replayer = CounterfactualReplayer()
    result = asyncio.run(replayer.replay_with_policy_disabled(
        policy_name=POLICY_TO_DISABLE,
        period_start=PERIOD_START,
        period_end=PERIOD_END,
    ))
    print(f'Hypothetical PnL: {result.hypothetical_pnl_usdc:.2f} USDC')
    print(f'Δ vs actual:      {result.delta_vs_actual:.2f} USDC')
    print(f'Decisions changed: {result.decisions_changed}')
    print(f'Wall time:         {result.wall_time_s:.1f}s')
except Exception as e:
    print(f'NO DATA or runtime error — populate the cold tier first. Detail: {e}')
